# Modelado con rezagos t-1 y t-2 — Predicción del rendimiento del maíz

## 0. Importación de librerías

In [1]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
from dotenv import load_dotenv
import mlflow
import mlflow.sklearn

In [2]:
cwd = Path.cwd().resolve()
try:
    HERE = next(c for c in [cwd, *cwd.parents]
                if (c / ".git").exists() or (c / ".dvc").exists())
except StopIteration:
    raise FileNotFoundError(
        "No encuentro la raíz del repo (.git/.dvc). "
        "Abre el notebook dentro del proyecto agropredict-colombia."
    )

DATOS_PROCESADOS = HERE / "data" / "processed"
ARCHIVO_FEATURES = DATOS_PROCESADOS / "features_rezago_1_2.parquet"

print(f"Repositorio: {HERE}")
print(f"Features disponibles: {ARCHIVO_FEATURES.exists()}")

Repositorio: /Users/Simon/Library/CloudStorage/OneDrive-SharedLibraries-ProductosAlimenticiosPascualS.A/Inteligencia Comercial - 5-Semestre/MIID-4304 Despliegue/Proyecto/agropredict-colombia
Features disponibles: True


In [3]:
load_dotenv(HERE / "config" / ".env", override=True)
MLFLOW_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5001")
MLFLOW_EXPERIMENT = os.getenv("MLFLOW_EXPERIMENT", "agropredict-maiz-rendimiento")

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

print(f"MLflow apuntando a: {MLFLOW_URI}")
print(f"Experimento: {MLFLOW_EXPERIMENT}")

2026/11/13 18:17:47 INFO mlflow.tracking.fluent: Experiment with name 'agropredict-pruebas-local' does not exist. Creating a new experiment.


MLflow apuntando a: http://localhost:5001
Experimento: agropredict-pruebas-local


In [2]:
from pathlib import Path
import pandas as pd

HERE = Path.home() / "agropredict-colombia"

datos = pd.read_parquet(
    HERE / "data" / "processed" / "features_rezago_1_2.parquet"
)

variables_modelo = [
    "semestre_b",
    "rendimiento_rezago_1",
    "rendimiento_rezago_2",
    "departamento",
    "sistema_productivo",
]

datos_modelado = datos.dropna(
    subset=variables_modelo + ["rendimiento"]
).copy()

print("=== DATOS PARA MODELADO ===")
print("Registros originales:", len(datos))
print("Registros con ambos rezagos:", len(datos_modelado))
print("Variables predictoras:", variables_modelo)

print("\n=== DISTRIBUCIÓN POR AÑO ===")
print(datos_modelado.groupby("anio").size())

=== DATOS PARA MODELADO ===
Registros originales: 14935
Registros con ambos rezagos: 10782
Variables predictoras: ['semestre_b', 'rendimiento_rezago_1', 'rendimiento_rezago_2', 'departamento', 'sistema_productivo']

=== DISTRIBUCIÓN POR AÑO ===
anio
2020    1839
2021    1916
2022    1810
2023    1769
2024    1713
2025    1735
dtype: int64


In [3]:
# Separación temporal: entrenamiento, validación y prueba

train = datos_modelado[
    datos_modelado["anio"] <= 2023
].copy()

val = datos_modelado[
    datos_modelado["anio"] == 2024
].copy()

test = datos_modelado[
    datos_modelado["anio"] == 2025
].copy()

X_train = train[variables_modelo]
y_train = train["rendimiento"]

X_val = val[variables_modelo]
y_val = val["rendimiento"]

X_test = test[variables_modelo]
y_test = test["rendimiento"]

print("=== PARTICIÓN TEMPORAL ===")
print(f"Entrenamiento: {len(X_train):,} registros")
print(f"Validación:    {len(X_val):,} registros")
print(f"Prueba final:  {len(X_test):,} registros")

print("\n=== VERIFICACIÓN TEMPORAL ===")
print("Años entrenamiento:", sorted(train["anio"].unique()))
print("Años validación:", sorted(val["anio"].unique()))
print("Años prueba:", sorted(test["anio"].unique()))

assert train["anio"].max() < val["anio"].min()
assert val["anio"].max() < test["anio"].min()

print("\nSeparación temporal verificada correctamente.")

=== PARTICIÓN TEMPORAL ===
Entrenamiento: 7,334 registros
Validación:    1,713 registros
Prueba final:  1,735 registros

=== VERIFICACIÓN TEMPORAL ===
Años entrenamiento: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Años validación: [np.int64(2024)]
Años prueba: [np.int64(2025)]

Separación temporal verificada correctamente.


In [4]:
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)
import numpy as np

# Modelo baseline: persistencia del semestre anterior
y_pred_persistencia = val["rendimiento_rezago_1"]

rmse = np.sqrt(mean_squared_error(y_val, y_pred_persistencia))
mae = mean_absolute_error(y_val, y_pred_persistencia)
r2 = r2_score(y_val, y_pred_persistencia)

print("=== BASELINE DE PERSISTENCIA — VALIDACIÓN 2024 ===")
print(f"Registros evaluados: {len(y_val):,}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"R²:   {r2:.4f}")

=== BASELINE DE PERSISTENCIA — VALIDACIÓN 2024 ===
Registros evaluados: 1,713
RMSE: 0.5672
MAE:  0.2209
R²:   0.8351
